# Cybercities: data download, part 2 (county GDP)

Click **Runtime → Run all**. Your browser downloads `cybercities_raw2.zip`; upload it back to Claude.

In [ ]:
import io, zipfile, requests, pandas as pd
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0 Safari/537.36"}

r = requests.get("https://apps.bea.gov/regional/zip/CAGDP1.zip", headers=HEADERS, timeout=300)
r.raise_for_status()
z = zipfile.ZipFile(io.BytesIO(r.content))
print("Files in the BEA zip:", z.namelist()[:10], "...")
name = [n for n in z.namelist() if "ALL_AREAS" in n and n.endswith(".csv")][0]
bea = pd.read_csv(z.open(name), dtype=str, encoding="latin-1")
bea["GeoFIPS"] = bea["GeoFIPS"].str.replace('"', "").str.strip()
bea["LineCode"] = bea["LineCode"].str.strip()
bea = bea[bea["LineCode"].isin(["1", "3"])]
bea.to_csv("bea_county_gdp.csv", index=False)
print(name, "->", len(bea), "rows kept")
print(bea[["GeoFIPS", "GeoName", "LineCode", "Description"]].head(6))

In [ ]:
with zipfile.ZipFile("cybercities_raw2.zip", "w", zipfile.ZIP_DEFLATED) as zz:
    zz.write("bea_county_gdp.csv")
from google.colab import files
files.download("cybercities_raw2.zip")